In [1]:
import importlib
from pathlib import Path
import numpy as np
import pandas as pd
import movie_recommender.data as movie_data

# Model selection

## Data Import

In [2]:
importlib.reload(movie_data)

build_movie_report = movie_data.build_movie_report
build_catalog_eda_frames = movie_data.build_catalog_eda_frames
build_user_eda_frames = movie_data.build_user_eda_frames
build_user_report = movie_data.build_user_report
prepare_movielens_frames = movie_data.prepare_movielens_frames
search_movies = movie_data.search_movies

DATA_DIR = Path("../movies-database")

movies        = pd.read_csv(DATA_DIR / "movies.csv")
ratings       = pd.read_csv(DATA_DIR / "ratings.csv")
tags          = pd.read_csv(DATA_DIR / "tags.csv")
links         = pd.read_csv(DATA_DIR / "links.csv")
genome_scores = pd.read_csv(DATA_DIR / "genome-scores.csv")
genome_tags   = pd.read_csv(DATA_DIR / "genome-tags.csv")

print("movies:       ", movies.shape)
print("ratings:      ", ratings.shape)
print("tags:         ", tags.shape)
print("links:        ", links.shape)
print("genome_scores:", genome_scores.shape)
print("genome_tags:  ", genome_tags.shape)

movies:        (62423, 3)
ratings:       (25000095, 4)
tags:          (1093360, 4)
links:         (62423, 3)
genome_scores: (15584448, 3)
genome_tags:   (1128, 2)


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Training spine  (25 M rows, stays lean)
# ─────────────────────────────────────────────────────────────────────────────

# Only merge the lightweight metadata you actually need per-rating.
# Keep genome completely out of this frame.
train = (
    ratings[["userId", "movieId", "rating", "timestamp"]]
    .merge(movies[["movieId", "genres"]], on="movieId", how="left")
)

# Lightweight datetime features for temporal models.
train["year_rated"]  = pd.to_datetime(train["timestamp"], unit="s").dt.year
train["month_rated"] = pd.to_datetime(train["timestamp"], unit="s").dt.month

print(train.shape)
print(train.dtypes)

(25000095, 7)
userId           int64
movieId          int64
rating         float64
timestamp        int64
genres          object
year_rated       int32
month_rated      int32
dtype: object


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Genre one-hot  (62 K × 19 cols)
# ─────────────────────────────────────────────────────────────────────────────
genre_dummies = (
    movies.set_index("movieId")["genres"]
    .str.get_dummies("|")
    .drop(columns=["(no genres listed)"], errors="ignore")
    .add_prefix("genre_")
    .astype("int8")           # saves memory vs int64
)
print(genre_dummies.shape)  # (62_423, 19)

(62423, 19)


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Genome: pivot to WIDE once (13 816 × 1 128 ≈ 120 MB in float32)
#           Store as a separate lookup — never join to ratings.
# ─────────────────────────────────────────────────────────────────────────────
tag_id_to_name = genome_tags.set_index("tagId")["tag"]

genome_wide = (
    genome_scores
    .pivot(index="movieId", columns="tagId", values="relevance")
    .astype("float32")
)
genome_wide.columns = [tag_id_to_name.get(t, str(t)) for t in genome_wide.columns]

print(genome_wide.shape)    # (13_816, 1_128)

(13816, 1128)


In [6]:
# --- BLOCK 0: Integer Encoding & Genre Alignment ---
# 1. Create 0-based integer indices for the embedding layers
user_ids  = np.sort(train["userId"].unique())
movie_ids = np.sort(train["movieId"].unique())

user_to_idx  = {uid: i for i, uid in enumerate(user_ids)}
movie_to_idx = {mid: i for i, mid in enumerate(movie_ids)}

N_USERS  = len(user_to_idx)
N_MOVIES = len(movie_to_idx)

train["user_idx"]  = train["userId"].map(user_to_idx).astype("int32")
train["movie_idx"] = train["movieId"].map(movie_to_idx).astype("int32")

print(f"Vocabulary -> Users: {N_USERS:,} | Movies: {N_MOVIES:,}")

# 2. Re-align the Genre Matrix so it matches the new integer indices
# (This prevents a NameError since you deleted the old Cell 11!)
idx_to_movie  = {v: k for k, v in movie_to_idx.items()}
all_movie_ids = [idx_to_movie[i] for i in range(N_MOVIES)]

genre_mat = (genre_dummies
             .reindex(all_movie_ids)
             .fillna(0)
             .values
             .astype("float32"))

N_GENRES = genre_mat.shape[1]

Vocabulary -> Users: 162,541 | Movies: 59,047


Block 1: The Leave-One-Out Split & Popularity Weighting
This cell drops the mean-centered ratings. It sorts your data by time, hides the very last movie each user watched to use as the test set, and calculates the inverse-frequency weights to penalize blockbusters.

In [7]:
import numpy as np
import pandas as pd
import random

# --- BLOCK 1: Temporal Sort & Rating Thresholding ---

# 1. Sort by User, then by timestamp
df_ratings_sorted = train.sort_values(['user_idx', 'timestamp']).copy()

# 2. Split by Rating to honor user satisfaction!
# >= 3.5 are "Likes" (Positives). < 3.5 are "Dislikes" (Hard Negatives)
df_likes = df_ratings_sorted[df_ratings_sorted['rating'] >= 3.5].copy()
df_dislikes = df_ratings_sorted[df_ratings_sorted['rating'] < 3.5].copy()

# 3. Extract the LAST "Liked" movie for each user for our Test Set
df_test_pos = df_likes.groupby('user_idx').tail(1).copy()

# 4. The rest of the "Likes" go into the Training Set
df_train_pos = df_likes.drop(df_test_pos.index).copy()

# 5. Set Targets
df_train_pos['target'] = 1.0  # They liked it
df_test_pos['target'] = 1.0   # They liked it
df_dislikes['target'] = 0.0   # Hard Negative: They watched it but didn't like it!

# 6. Calculate Dynamic Weights based on POSITIVE interactions to fix Popularity Bias
movie_counts = df_train_pos['movie_idx'].value_counts()
raw_weights = 1.0 / np.sqrt(movie_counts)
normalized_weights = (raw_weights / raw_weights.mean()).to_dict()

df_train_pos['weight'] = df_train_pos['movie_idx'].map(normalized_weights).fillna(1.0)
df_dislikes['weight'] = df_dislikes['movie_idx'].map(normalized_weights).fillna(1.0)

print(f"Total 'Likes' (Train): {len(df_train_pos)}")
print(f"Total 'Dislikes' (Hard Negatives): {len(df_dislikes)}")

Total 'Likes' (Train): 15467715
Total 'Dislikes' (Hard Negatives): 9369966


Block 2: Negative Sampling for Train & Test
A model can't learn what you want to watch if it doesn't know what you don't want to watch. This cell generates fake "0" interactions for training, and sets up the "1 Positive vs 99 Negatives" test for evaluation.

In [8]:
import random
import pandas as pd
from tqdm import tqdm

# --- BLOCK 2: Fast Negative Sampling with Progress Bar ---

all_movie_list = list(train['movie_idx'].dropna().unique())
user_interacted = df_ratings_sorted.groupby('user_idx')['movie_idx'].apply(set).to_dict()

# Create a "Hard Decoy" pool of the top 2000 most popular movies
top_2000_movies = df_train_pos['movie_idx'].value_counts().head(2000).index.tolist()

# --- TRAIN SET SAMPLED NEGATIVES ---
train_negatives = []

# Use tqdm to wrap the loop — this will show the progress bar!
for _, row in tqdm(df_train_pos.iterrows(), total=len(df_train_pos), desc="Sampling negatives"):
    u = row['user_idx']
    interacted = user_interacted[u]

    negatives = set()
    # Pushed to 7 negatives to force the Neural Network to learn tighter boundaries
    while len(negatives) < 7:
        m_neg = random.choice(all_movie_list)
        if m_neg not in interacted:
            negatives.add(m_neg)

    for m_neg in negatives:
        train_negatives.append({'user_idx': u, 'movie_idx': m_neg, 'target': 0.0, 'weight': 1.0})

# Combine
df_train = pd.concat([
    df_train_pos[['user_idx', 'movie_idx', 'target', 'weight']],
    df_dislikes[['user_idx', 'movie_idx', 'target', 'weight']],
    pd.DataFrame(train_negatives)
])
df_train = df_train.sample(frac=1).reset_index(drop=True)


# --- TEST SET DECOYS (ALL USERS - OPTIMIZED) ---
test_eval_data = []

# Convert the top 2000 list into a set ONCE for incredibly fast math
top_2000_set = set(top_2000_movies)

# Notice we removed the .head()! This will process ALL 162,000 users blazing fast.
for _, row in tqdm(df_test_pos.iterrows(), total=len(df_test_pos), desc="Building ALL test decoys"):
    u = int(row['user_idx'])
    m_pos = int(row['movie_idx'])
    interacted = user_interacted[u]

    # 1. The Set Difference: Instantly find all available decoys
    available_decoys = top_2000_set - interacted

    # 2. The Super User Safety Check
    # If they watched almost everything in the top 2000, just grab whatever is left.
    num_decoys = min(99, len(available_decoys))

    # 3. Instantly sample the exact number we need without a while loop
    decoy_negatives = random.sample(list(available_decoys), num_decoys)

    items_to_score = [m_pos] + decoy_negatives
    test_eval_data.append({'user_idx': u, 'items': items_to_score})

print(f"Final Train Set Size: {len(df_train)}")
print(f"Successfully built decoys for {len(test_eval_data)} users!")
print(f"Ready for HARD HitRate@10 Eval.")

Building ALL test decoys: 100%|██████████| 162414/162414 [00:44<00:00, 3611.99it/s] 

Final Train Set Size: 133111686
Successfully built decoys for 162414 users!
Ready for HARD HitRate@10 Eval.


Block 3: The SVD Baseline
We run Matrix Factorization first to set the benchmark. It uses a pure Dot Product.

In [9]:
from scipy.sparse import csr_matrix, vstack
from scipy.sparse.linalg import svds
import numpy as np

print("Aligning Genome Matrix...")
genome_mat = (genome_wide
              .reindex(all_movie_ids)
              .fillna(0)
              .values
              .astype("float32"))

# We give it a massive brain (k=256) to handle the 1,128 Genome tags
K_GRID = [128, 256]
BETA_GRID = [0.1, 0.5] # Genome weight

best_hr = 0
best_params = {}

print("Building the Sparse Matrix 'R'...")
rows = df_train_pos['user_idx'].values
cols = df_train_pos['movie_idx'].values

# THE FIX: Go back to np.ones! Let the model guess blockbusters so it can pass the test!
data = np.ones(len(rows), dtype=np.float32)

R = csr_matrix((data, (rows, cols)), shape=(N_USERS, N_MOVIES), dtype=np.float32)

print("Starting Unchained Hybrid SVD Grid Search...")

for beta in BETA_GRID:
    # Transpose the Genome matrix (we skip genres because Genome already includes them and is 100x better)
    Gen_sparse = csr_matrix((genome_mat * beta).T)

    # THE HYBRID STACK: Users + 1,128 Genome Tags
    R_aug = vstack([R, Gen_sparse], format="csr")

    for k in K_GRID:
        # Note: k=256 will take a minute or two to calculate!
        U, s, Vt = svds(R_aug, k=k)

        # Slice U to only grab the actual User vectors
        U_users = U[:N_USERS, :]
        U_sig = np.dot(U_users, np.diag(s))

        # Evaluate HitRate@10
        hits = 0
        for data_eval in test_eval_data:
            u = data_eval['user_idx']
            items = data_eval['items']

            scores = np.dot(U_sig[u], Vt[:, items])
            top_10_idx = np.argsort(scores)[::-1][:10]

            if 0 in top_10_idx:
                hits += 1

        hr10 = hits / len(test_eval_data)
        print(f"Grid Search -> k={k:3}, genome_w={beta:3.1f} | HitRate@10: {hr10:.4f}")

        if hr10 > best_hr:
            best_hr = hr10
            best_params = {'k': k, 'beta': beta}
            best_Vt = Vt

print(f"\n🏆 Best Params: {best_params} | Best HitRate: {best_hr:.4f}")

Aligning Genome Matrix...
Building the Sparse Matrix 'R'...
Starting Unchained Hybrid SVD Grid Search...
Grid Search -> k=128, genome_w=0.1 | HitRate@10: 0.5044
Grid Search -> k=256, genome_w=0.1 | HitRate@10: 0.4370
Grid Search -> k=128, genome_w=0.5 | HitRate@10: 0.5052
Grid Search -> k=256, genome_w=0.5 | HitRate@10: 0.4376

🏆 Best Params: {'k': 128, 'beta': 0.5} | Best HitRate: 0.5052


Block 4: The NCF Challenger (BCE & Weights)
Now we train the deep learning model. Notice the custom PyTorch Batcher, the BCEWithLogitsLoss (for predicting probabilities instead of stars), and the dynamic weighting applied to the loss function.

In [10]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")

PyTorch Version: 2.5.1+cu121
CUDA Available:  True
Device Name:     NVIDIA GeForce RTX 4070


In [11]:
import torch
import gc

# Tell Python to collect any unreferenced variables
gc.collect()

# Force PyTorch to empty its "Ghost" cache back to the system
torch.cuda.empty_cache()

# Check how much VRAM is currently being used
allocated = torch.cuda.memory_allocated() / (1024 ** 3)
reserved = torch.cuda.memory_reserved() / (1024 ** 3)
print(f"GPU Memory Allocated: {allocated:.2f} GB")
print(f"GPU Memory Reserved:  {reserved:.2f} GB")

GPU Memory Allocated: 0.00 GB
GPU Memory Reserved:  0.00 GB


In [12]:
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import StepLR
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm
import gc
import numpy as np
import random

# --- Setup ---
gc.collect()
torch.cuda.empty_cache()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ensure matrices are aligned
print("Aligning Genome Matrix...")
genome_mat = (genome_wide.reindex(all_movie_ids).fillna(0).values.astype("float32"))

genre_tensor_all = torch.tensor(genre_mat, dtype=torch.float32).to(DEVICE)
genome_tensor_all = torch.tensor(genome_mat, dtype=torch.float32).to(DEVICE)
N_GENOME = genome_tensor_all.shape[1]

# 1. The BPR-Ready Batcher
class GPUBatcherBPR:
    def __init__(self, pos_df, all_movie_ids, user_interacted_map, batch_size):
        self.u = torch.tensor(pos_df['user_idx'].values, dtype=torch.long, device=DEVICE)
        self.pos_m = torch.tensor(pos_df['movie_idx'].values, dtype=torch.long, device=DEVICE)
        self.all_movie_ids = all_movie_ids
        self.user_interacted_map = user_interacted_map
        self.n, self.bs = len(pos_df), batch_size

    def __iter__(self):
        idx = torch.randperm(self.n, device=DEVICE)
        for i in range(0, self.n, self.bs):
            j = idx[i:i + self.bs]
            batch_u = self.u[j]
            batch_pos_m = self.pos_m[j]

            # Negative Sampling
            batch_neg_m = []
            for u in batch_u.cpu().numpy():
                while True:
                    # 80% chance to pick from the "Hard" top 2000 list
                    if random.random() < 0.8:
                        neg = random.choice(top_2000_movies)
                    else:
                        neg = random.choice(self.all_movie_ids)

                    if neg not in self.user_interacted_map[u]:
                        batch_neg_m.append(neg)
                        break

            batch_neg_m = torch.tensor(batch_neg_m, dtype=torch.long, device=DEVICE)
            yield batch_u, batch_pos_m, batch_neg_m

    def __len__(self):
        return (self.n + self.bs - 1) // self.bs

# 2. Model
class NeuralCF_Genome(nn.Module):
    def __init__(self, n_users, n_movies, n_genres=19, n_genome=1128, n_factors=128, dropout_rate=0.2):
        super().__init__()
        self.u_gmf = nn.Embedding(n_users, n_factors)
        self.m_gmf = nn.Embedding(n_movies, n_factors)
        self.u_mlp = nn.Embedding(n_users, n_factors)
        self.m_mlp = nn.Embedding(n_movies, n_factors)
        self.u_bias = nn.Embedding(n_users, 1)
        self.m_bias = nn.Embedding(n_movies, 1)
        self.genre_proj = nn.Linear(n_genres, n_factors, bias=False)
        self.genome_proj = nn.Linear(n_genome, n_factors, bias=False)

        input_dim = (2 * n_factors) + n_genres + n_genome
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(128, 64), nn.ReLU()
        )
        self.output_layer = nn.Linear(n_factors + 64, 1)

    def forward(self, u, m):
        g, gen = genre_tensor_all[m], genome_tensor_all[m]
        # Boost genome influence
        gmf_out = self.u_gmf(u) * (self.m_gmf(m) + self.genre_proj(g) + (self.genome_proj(gen) * 2.0))
        mlp_out = self.mlp(torch.cat([self.u_mlp(u), self.m_mlp(m), g, gen], dim=1))
        logits = self.output_layer(torch.cat([gmf_out, mlp_out], dim=1))
        return (logits + self.u_bias(u) + self.m_bias(m)).squeeze(1)

# 3. Training
train_dl = GPUBatcherBPR(df_train_pos, all_movie_list, user_interacted, batch_size=8192)
model_ncf = NeuralCF_Genome(N_USERS, N_MOVIES, n_genres=N_GENRES, n_genome=N_GENOME).to(DEVICE)
optimiser = torch.optim.Adam(model_ncf.parameters(), lr=0.003)
scheduler = StepLR(optimiser, step_size=6, gamma=0.5)
scaler = GradScaler('cuda')

def bpr_loss(pos_scores, neg_scores):
    return -torch.mean(torch.nn.functional.logsigmoid(pos_scores - neg_scores))

print("🚀 Training NCF with BPR Loss...")
for epoch in range(1, 16):
    model_ncf.train()
    total_loss = 0
    for u, pos_m, neg_m in tqdm(train_dl, desc=f"Epoch {epoch}"):
        optimiser.zero_grad()
        with autocast(device_type='cuda'):
            pos_scores = model_ncf(u, pos_m)
            neg_scores = model_ncf(u, neg_m)
            loss = bpr_loss(pos_scores, neg_scores)
        scaler.scale(loss).backward()
        scaler.step(optimiser)
        scaler.update()
        total_loss += loss.item()

    # Evaluation
    model_ncf.eval()
    hits = 0
    with torch.no_grad():
        for data in tqdm(test_eval_data, desc="Evaluating"):
            u = torch.tensor([data['user_idx']] * len(data['items']), dtype=torch.long, device=DEVICE)
            m = torch.tensor(data['items'], dtype=torch.long, device=DEVICE)
            scores = model_ncf(u, m).cpu().numpy()
            if 0 in np.argsort(scores)[::-1][:10]:
                hits += 1
    scheduler.step()
    print(f"✅ Epoch {epoch} | Loss: {total_loss/len(train_dl):.4f} | HR@10: {hits/len(test_eval_data):.4f}")

Aligning Genome Matrix...
🚀 Training NCF with BPR Loss...


Epoch 1:   0%|          | 0/1889 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/162414 [00:00<?, ?it/s]

✅ Epoch 1 | Loss: 0.3285 | HR@10: 0.5292


Epoch 2:   0%|          | 0/1889 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/162414 [00:00<?, ?it/s]

✅ Epoch 14 | Loss: 0.1125 | HR@10: 0.6973


Epoch 15:   0%|          | 0/1889 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/162414 [00:00<?, ?it/s]

✅ Epoch 15 | Loss: 0.1109 | HR@10: 0.6982


In [18]:
np.save("../docker-app/services/mlengine/svd_item_matrix.npy", Vt)

In [17]:
import pickle
import torch

# 1. Pre-calculate the Normalized Ratings (Quality Score)
movie_avg_ratings = ratings.groupby('movieId')['rating'].mean()
full_avg_ratings = torch.full((N_MOVIES,), 3.0) # Default to 3.0
for mid, idx in movie_to_idx.items():
    if mid in movie_avg_ratings:
        full_avg_ratings[idx] = movie_avg_ratings[mid]
norm_ratings = (full_avg_ratings - 0.5) / 4.5

# 2. Pre-calculate the Log Popularity (Penalty Score)
counts = df_train_pos['movie_idx'].value_counts().sort_index()
full_pop = torch.zeros(N_MOVIES)
full_pop[torch.tensor(counts.index.values)] = torch.tensor(counts.values, dtype=torch.float32)
log_pop = torch.log(full_pop + 1)

# 3. Export everything your API needs!
metadata = {
    "N_USERS": N_USERS,
    "N_MOVIES": N_MOVIES,
    "movie_to_idx": movie_to_idx,
    "idx_to_movie": {v: k for k, v in movie_to_idx.items()},
    "N_GENRES": N_GENRES,
    "N_GENOME": N_GENOME,
    "all_genres_tensor": genre_tensor_all.cpu(),
    "genome_tensor_all": genome_tensor_all.cpu(), # <-- FIXED NAME
    "norm_ratings": norm_ratings.cpu(),           # <-- ADDED
    "log_pop": log_pop.cpu()                      # <-- ADDED
}

# Save metadata
with open("../docker-app/services/mlengine/ncf_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

# Save weights
torch.save(model_ncf.state_dict(), "../docker-app/services/mlengine/ncf_weights.pth")

print("✅ Model and Metadata (including Ratings and Popularity) successfully exported!")

✅ Model and Metadata (including Ratings and Popularity) successfully exported!


In [16]:
def recommend_vs_showdown(liked_movie_ids, svd_matrix, model_ncf, movie_to_idx, idx_to_movie, top_n=10):
    movie_titles = movies.set_index('movieId')['title'].to_dict()
    # 1. Print inputs
    print("🎬 YOU SELECTED THESE MOVIES:")
    for mid in liked_movie_ids:
        print(f"   - {movie_titles.get(mid, 'Unknown ID')}")
    print("\n" + "="*100)

    liked_indices = [movie_to_idx[mid] for mid in liked_movie_ids if mid in movie_to_idx]
    if not liked_indices: return "No valid movies found."

    # 2. Pre-calculate Avg Ratings (Quality Signal)
    movie_avg_ratings = ratings.groupby('movieId')['rating'].mean()
    full_avg_ratings = torch.full((N_MOVIES,), 3.0, device=DEVICE)
    for mid, idx in movie_to_idx.items():
        if mid in movie_avg_ratings:
            full_avg_ratings[idx] = movie_avg_ratings[mid]

    # Normalize ratings to 0-1 scale for balanced weighting
    norm_ratings = (full_avg_ratings - 0.5) / 4.5

    # ==========================================
    # MODEL 1: SVD Baseline
    # ==========================================
    svd_user_vec = np.sum(svd_matrix[:, liked_indices], axis=1)
    svd_scores = np.dot(svd_user_vec, svd_matrix)
    svd_scores[liked_indices] = -np.inf
    svd_top_10 = np.argsort(svd_scores)[::-1][:top_n]

    # ==========================================
    # MODEL 2: NCF Multi-Objective (Quality + Genome)
    # ==========================================
    model_ncf.eval()
    with torch.no_grad():
        liked_tensor = torch.tensor(liked_indices, dtype=torch.long, device=DEVICE)

        # A. Genome Similarity
        proj_liked = model_ncf.genome_proj(genome_tensor_all[liked_tensor].mean(dim=0).unsqueeze(0))
        proj_all = model_ncf.genome_proj(genome_tensor_all)
        genome_sim = torch.matmul(proj_liked, proj_all.t()).squeeze()

        # B. Final Weighted Score: 60% Genome Fit, 40% Rating Quality
        ncf_scores = (0.6 * genome_sim) + (0.4 * norm_ratings)

        # C. Popularity Penalty
        pop_penalty = 0.3
        counts = df_train_pos['movie_idx'].value_counts().sort_index()
        full_pop = torch.zeros(N_MOVIES, device=DEVICE)
        full_pop[torch.tensor(counts.index.values, device=DEVICE)] = torch.tensor(counts.values, dtype=torch.float32, device=DEVICE)
        ncf_scores = ncf_scores - (pop_penalty * torch.log(full_pop + 1))

        ncf_scores = ncf_scores.cpu().numpy()
        ncf_scores[liked_indices] = -np.inf
        ncf_top_10 = np.argsort(ncf_scores)[::-1][:top_n]

    # ==========================================
    # RESULTS
    # ==========================================
    print("🍿 BASED ON YOUR LIKES (GENOME + QUALITY):")
    print(f"{'Rank':<5} | {'SVD Baseline':<45} | {'NCF Multi-Objective':<45}")
    print("-" * 100)
    for rank in range(top_n):
        svd_t = movie_titles.get(idx_to_movie[svd_top_10[rank]], "Unknown")[:42]
        ncf_t = movie_titles.get(idx_to_movie[ncf_top_10[rank]], "Unknown")[:42]
        print(f"{rank+1:<5} | {svd_t:<45} | {ncf_t:<45}")

# Run the verification
my_favorites = [166455, 72226, 139130]
recommend_vs_showdown(my_favorites, Vt, model_ncf, movie_to_idx, idx_to_movie)

🎬 YOU SELECTED THESE MOVIES:
   - Lego DC Comics Superheroes: Justice League - Gotham City Breakout (2016)
   - Fantastic Mr. Fox (2009)
   - Afro Samurai (2007)

🍿 BASED ON YOUR LIKES (GENOME + QUALITY):
Rank  | SVD Baseline                                  | NCF Multi-Objective                          
----------------------------------------------------------------------------------------------------
1     | Royal Tenenbaums, The (2001)                  | Princess Mononoke (Mononoke-hime) (1997)     
2     | Rushmore (1998)                               | Nausicaä of the Valley of the Wind (Kaze n   
3     | Moonrise Kingdom (2012)                       | My Neighbor Totoro (Tonari no Totoro) (198   
4     | Grand Budapest Hotel, The (2014)              | Akira (1988)                                 
5     | Life Aquatic with Steve Zissou, The (2004)    | Laputa: Castle in the Sky (Tenkû no shiro    
6     | Darjeeling Limited, The (2007)                | Lupin III: The Castle Of C